# 업종은 어디에도 없었다 — 사람 손 18번과 종가 역추적

> `notebooks/01-데이터수집/07.업종은어디에도없었다.ipynb` · 2026-09-03 · 이동원
> 기능명세 [`docs/기능명세/version1.3/업종_스냅샷_공급.md`] ·
> 가이드 [`docs/데이터파트/version3.2/직접수집_가이드_업종분류현황.md`]

---

## 이 노트북이 답하는 것

> **"업종지수 → 그 업종의 개별 종목 → 시총 1·2위" 연결을 왜 못 만들었고, 어떻게 만들었나**

모델 파트 2안(2주 MVP)은 *KOSPI200 방향 → 시총 상위 10개 업종 → 업종마다 상위 4~5 종목* 이다.
팀원은 *"`sector` 만 채워 주면 된다"* 고 했다. 채우려고 보니 **채울 출처가 없었다.**

In [1]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name.startswith("0") else Path.cwd()
sys.path.insert(0, str(ROOT))

from supply import sector, universe  # noqa: E402

conn = sqlite3.connect(f"file:{(ROOT / 'data/krx_cache.db').as_posix()}?mode=ro", uri=True)
pd.set_option("display.width", 120)
print("DB 읽기 전용으로 연결")

DB 읽기 전용으로 연결


---

## 1. `sector` 칸에는 무엇이 들어 있었나

KRX OpenAPI 의 `SECT_TP_NM` 을 그대로 담은 칸이다. 이름만 보고 업종이라고 생각했다.

In [2]:
for d in ["20100104", "20200102", "20260901"]:
    rows = conn.execute(
        "SELECT market, COALESCE(NULLIF(sector, ''), '(빈 값)') s, COUNT(*) FROM daily_price "
        "WHERE bas_dd = ? GROUP BY 1, 2 ORDER BY 1, 3 DESC", (d,)).fetchall()
    print(f"[{d}]")
    for m in ("KOSPI", "KOSDAQ"):
        값들 = [(s, n) for mk, s, n in rows if mk == m]
        print(f"  {m:<7}", " · ".join(f"{s}({n})" for s, n in 값들[:5]),
              "…" if len(값들) > 5 else "")

[20100104]
  KOSPI   (빈 값)(925) 
  KOSDAQ  (빈 값)(1036) 
[20200102]
  KOSPI   (빈 값)(916) 
  KOSDAQ  중견기업부(479) · 우량기업부(379) · 벤처기업부(295) · 기술성장기업부(87) · 관리종목(소속부없음)(57) …
[20260901]
  KOSPI   (빈 값)(943) 
  KOSDAQ  중견기업부(512) · 우량기업부(466) · 벤처기업부(342) · 기술성장기업부(255) · 관리종목(소속부없음)(123) …


**KOSPI 는 전부 빈 값이고, KOSDAQ 은 소속부다.** 산업 업종이 아니다.

`index_price` 에는 업종지수와 시가총액이 있다. 하지만 **어느 종목이 그 지수에 드는지**는 없다.

In [3]:
caps = pd.read_sql_query(
    "SELECT index_name, market_cap FROM index_price "
    "WHERE bas_dd = '20240102' AND index_class = 'KOSPI' AND index_name NOT LIKE '코스피%' "
    "ORDER BY market_cap DESC", conn)
caps["market_cap_조"] = (caps["market_cap"] / 1e12).round(1)
print(f"20240102 KOSPI 업종지수 {len(caps)}종 — 시총 상위 10")
print(caps[["index_name", "market_cap_조"]].head(10).to_string(index=False))

20240102 KOSPI 업종지수 21종 — 시총 상위 10
index_name  market_cap_조
        제조        1526.4
      전기전자         886.7
        금융         270.5
     일반서비스         177.3
   운송장비·부품         170.2
        화학         160.8
        제약         124.6
        금속          72.5
        유통          69.2
     기계·장비          48.1


---

## 2. 되는 길은 있었다 — 그러나 써도 되는 길은 아니었다

pykrx 의 `업종분류현황` 이 종목마다 업종을 준다. **로그인하면** 2010년까지 열린다
(pykrx 자신의 "2014-05-01 이전 불가" 주석은 래퍼가 스스로 막은 것일 뿐이었다).

그런데 [KRX Data Marketplace 홈페이지 이용약관](https://data.krx.co.kr/contents/MDC/INFO/informationController/MDCINFO003.cmd)
(2026-08-29 시행):

> **제10조 ② 자동화 수단을 이용하여 정보를 무단 수집·복제·배포하는 행위** — 금지
> **제12조 ②** 사전 허락 없이 복제·배포 — 금지

그래서 코드로 받지 않았다. **사람이 화면에서 내려받는 것**은 약관이 의도한 사용이다.
연 1회 스냅샷이면 클릭 18번이다. 이 노트북은 그 18장이 들어온 **뒤**의 이야기다.

---

## 3. 🔴 화면 CSV 에는 기준일이 없다

`data_2708_20260903.csv` — 파일명의 날짜는 조회일이 아니라 **내려받은 날**이다.
2015-01-02 를 조회해 오늘 받으면 `20260903` 이 붙는다. 사람에게 이름을 고쳐 적게 하면 틀린다.

그래서 `scripts/normalize_manual.py` 가 **종가 30종을 `daily_price` 와 맞춰 날짜를 되짚는다.**
30종의 종가가 동시에 같은 거래일은 하나뿐이다. 되짚은 결과는 `MANIFEST.json` 에 남는다.

In [4]:
manifest = json.loads((ROOT / "data/manual/normalized/MANIFEST.json").read_text(encoding="utf-8"))
되짚음 = [(k, v["inferred"]) for k, v in manifest["files"].items() if "inferred" in v]
print(f"되짚은 파일 {len(되짚음)}개")
for k, v in 되짚음[:4]:
    print(f"  {k:<40} → {v['bas_dd']} {v['market']}  ({v['how']})")
print("  …")

되짚은 파일 18개
  sector/업종분류현황_KOSPI_20100104.csv         → 20100104 KOSPI  (종가 30/30종이 daily_price 와 일치)
  sector/업종분류현황_KOSPI_20110103.csv         → 20110103 KOSPI  (종가 30/30종이 daily_price 와 일치)
  sector/업종분류현황_KOSPI_20120102.csv         → 20120102 KOSPI  (종가 30/30종이 daily_price 와 일치)
  sector/업종분류현황_KOSPI_20130102.csv         → 20130102 KOSPI  (종가 30/30종이 daily_price 와 일치)
  …


---

## 4. 들어온 것 — 반입 엔진이 통과시킨 18장

규격 `ingest/inbox/schemas/sector.json` 으로 검사했다. 격리 0.

In [5]:
snaps = sector.snapshots()
days = sorted(snaps["bas_dd"].unique())
행수 = snaps.groupby("bas_dd").size()
print(f"스냅샷 {len(days)}장 · {len(snaps):,}행 · 시장 {sorted(snaps['market'].unique())}")
print(" ".join(f"{d}:{행수[d]}" for d in days))

스냅샷 18장 · 16,650행 · 시장 ['KOSPI']
20100104:925 20110103:927 20120102:938 20130102:930 20140102:917 20150102:899 20160104:887 20170102:894 20180102:887 20190102:901 20200102:916 20210104:917 20220103:942 20230102:943 20240102:953 20240701:955 20250102:961 20260102:958


---

## 5. 🔴 행 수가 아니라 값으로 — 16,650행 전량 대조

되짚은 기준일이 하루라도 어긋났으면 종가가 안 맞는다. 같은 날을 두 번 들여 덮어썼어도
행 수는 그대로다. 그래서 **모든 행**의 종가·시가총액·시장을 `daily_price` 와 맞댄다.

In [6]:
acc = pd.read_sql_query("SELECT payload FROM inbox_accepted WHERE kind = 'sector'", conn)
rows = pd.DataFrame([json.loads(p) for p in acc["payload"]])
rows["close"] = rows["close"].astype("Int64")
rows["market_cap"] = rows["market_cap"].astype("Int64")
price = pd.read_sql_query(
    "SELECT bas_dd, code, close, market_cap, market FROM daily_price "
    f"WHERE bas_dd IN ({','.join('?' * len(days))})", conn, params=days)
m = rows.merge(price, on=["bas_dd", "code"], how="left", suffixes=("", "_db"))
있음 = m["close_db"].notna()
print(f"스냅샷 {len(m):,}행 · daily_price 에 없는 행 {int((~있음).sum()):,}")
print(f"종가 다른 행     {int(((m['close'] != m['close_db']) & 있음).sum()):,}")
print(f"시가총액 다른 행 {int(((m['market_cap'] != m['market_cap_db']) & 있음).sum()):,}")
print(f"시장 다른 행     {int(((m['market'] != m['market_db']) & 있음).sum()):,}")

스냅샷 16,650행 · daily_price 에 없는 행 0
종가 다른 행     0
시가총액 다른 행 0
시장 다른 행     0


**하나도 안 틀렸다.** 되짚기가 18장 전부 맞았다는 뜻이다.

---

## 6. 업종 체계는 바뀐다 — 그리고 내 전제가 틀렸다

가이드를 쓸 때 "2024-07-01 = 체계 개편일" 로 적었다. `IT 서비스`·`부동산`·`오락·문화`
**지수**는 정말 그날 생겼다. 그런데 그날 스냅샷의 **종목 분류**는 어땠나.

In [7]:
이름들 = {d: set(snaps.loc[snaps["bas_dd"] == d, "sector_nm"]) for d in days}
표시 = [f"{d[:4]}{'/07' if d[4:6] == '07' else ''}:{len(이름들[d])}" for d in days]
print("업종 수:", " ".join(표시))
for 업종 in ["IT 서비스", "부동산", "오락·문화"]:
    처음 = next(d for d in days if 업종 in 이름들[d])
    print(f"  {업종:<8} 처음 나타난 스냅샷: {처음}")
print("  20240102 → 20240701 사라진 업종:", sorted(이름들["20240102"] - 이름들["20240701"]))

for code_, nm in [("035420", "NAVER"), ("033780", "KT&G"), ("005930", "삼성전자")]:
    seq = snaps[snaps["code"] == code_].sort_values("bas_dd")
    바뀜 = [f"{r.bas_dd}:{r.sector_nm}" for i, r in enumerate(seq.itertuples())
           if i == 0 or r.sector_nm != seq.iloc[i - 1]["sector_nm"]]
    print(f"  {nm:<6}", " → ".join(바뀜))

업종 수: 2010:24 2011:24 2012:24 2013:24 2014:24 2015:24 2016:24 2017:24 2018:24 2019:24 2020:24 2021:24 2022:24 2023:24 2024:24 2024/07:23 2025:26 2026:26
  IT 서비스   처음 나타난 스냅샷: 20250102
  부동산      처음 나타난 스냅샷: 20250102
  오락·문화    처음 나타난 스냅샷: 20250102
  20240102 → 20240701 사라진 업종: ['광업']
  NAVER  20100104:일반서비스 → 20250102:IT 서비스
  KT&G   20100104:기타제조 → 20250102:음식료·담배
  삼성전자   20100104:전기·전자


지수가 생긴 날과 종목이 옮겨진 날은 **다른 날**이었다. 2024-07-01 스냅샷은 옛 체계
(광업만 사라져 23종)이고, NAVER 가 `IT 서비스` 로 옮긴 것은 2025-01-02 부터다.
바뀐 날은 2024-07-02 ~ 2025-01-02 사이 — 좁히려면 한 장 더 받으면 된다.

as-of 규칙("그날 이전 가장 최근 스냅샷") 덕에 이 지연은 **미래참조가 아니라 늦음**이다.
늦는 쪽으로 틀리는 것은 성능을 부풀리지 않는다.

---

## 7. 붙여 보기 — `supply.sector.attach_industry`

시세 표에 행마다 **그 행의 날짜 이전 가장 최근 스냅샷**을 붙인다. `as_of` 가 없으면 못 지난다.

In [8]:
kospi = pd.read_sql_query(
    "SELECT bas_dd, code FROM daily_price WHERE market = 'KOSPI' AND bas_dd <= '20240830'", conn)
붙인 = sector.attach_industry(kospi, as_of="2026-09-03")
붙인["연도"] = 붙인["bas_dd"].str[:4]
덮임 = 붙인.groupby("연도")["industry"].apply(lambda s: s.notna().mean())
print(f"KOSPI 개발구간 {len(붙인):,}행 · industry 채움 {붙인['industry'].notna().mean():.1%}")
print("연도별:", " ".join(f"{y}:{v:.0%}" for y, v in 덮임.items()))

KOSPI 개발구간 3,315,889행 · industry 채움 99.0%
연도별: 2010:98% 2011:98% 2012:99% 2013:99% 2014:100% 2015:99% 2016:99% 2017:99% 2018:99% 2019:99% 2020:99% 2021:99% 2022:100% 2023:99% 2024:100%


연초 스냅샷 뒤에 상장한 종목은 그해 말까지 `industry` 가 비는데, 그것이 채움이 100% 가
아닌 이유다. **모르는 값을 지어내지 않는다** — 다음 해 첫 스냅샷에서 채워진다.

---

## 8. 2안을 실제로 돌려 본다 — 업종 상위 10 → 업종별 시총 상위 5

이 셀이 대조표의 잘못 둘을 잡았다.

- `보험`·`증권` 을 `금융` 으로 올렸더니 **보험 지수에 붙는 종목이 0** 이었다 → 둘은 자기 지수가 있다
- `제조`(1,526조) 는 제조업 전부의 합 — 넣으면 그 자리에 `기타제조` 소형주가 뽑혔다 → 상위 묶음은 뺀다

In [9]:
day, as_of = "20240102", "2024-01-04"
묶음빼고 = caps[~caps["index_name"].isin(sector.UMBRELLA_INDICES)]
후보지수 = 묶음빼고.nlargest(10, "market_cap")
보통주 = universe.common_stocks(day, as_of=as_of)
보통주 = sector.attach_industry(보통주, as_of=as_of)
보통주["index_name"] = 보통주["industry"].map(sector.index_name_for)
뽑힌 = []
for _, r in 후보지수.iterrows():
    top5 = 보통주[보통주["index_name"] == r["index_name"]].nlargest(5, "market_cap")
    뽑힌.append(top5)
    print(f"{r['index_name']:<10} {r['market_cap'] / 1e12:6.1f}조 → {', '.join(top5['name'])}")
총 = pd.concat(뽑힌)
print(f"\n→ {len(총)}종 · 묶음 지수 {sorted(sector.UMBRELLA_INDICES)} 제외"
      f" · industry 없는 보통주 {int(보통주['industry'].isna().sum())}/{len(보통주)}")

전기전자        886.7조 → 삼성전자, SK하이닉스, LG에너지솔루션, 삼성SDI, 포스코퓨처엠
일반서비스       177.3조 → NAVER, 카카오, 삼성에스디에스, 포스코DX, 하이브
운송장비·부품     170.2조 → 현대차, 기아, 현대모비스, HD현대중공업, 한화오션
화학          160.8조 → LG화학, SK이노베이션, 아모레퍼시픽, S-Oil, 한화솔루션
제약          124.6조 → 삼성바이오로직스, 셀트리온, SK바이오사이언스, 유한양행, 한미약품
금속           72.5조 → POSCO홀딩스, 고려아연, 현대제철, 삼아알미늄, TCC스틸
유통           69.2조 → 삼성물산, 포스코인터내셔널, 호텔신라, GS리테일, BGF리테일
기계·장비        48.1조 → 두산에너빌리티, 두산로보틱스, 한미반도체, 두산밥캣, 한온시스템
운송·창고        46.5조 → HMM, 대한항공, 현대글로비스, 한진칼, CJ대한통운
보험           41.6조 → 삼성생명, 삼성화재, DB손해보험, 현대해상, 한화생명

→ 50종 · 묶음 지수 ['금융', '제조'] 제외 · industry 없는 보통주 0/839


---

## 9. 팀원이 받는 것

`daily_price_dev.parquet` 에 세 칸이 **새로** 붙는다. `sector`(소속부)는 그대로 둔다 —
한 칸에 두 뜻을 섞지 않는다.

| 칸 | 뜻 |
|---|---|
| `industry` | KRX 업종명 그대로 (`전기·전자` · `IT 서비스` …) |
| `industry_bas_dd` | 어느 스냅샷에서 왔나 |
| `industry_known_at` | 그 스냅샷을 언제부터 알 수 있었나 (다음 거래일) |

업종지수와 붙일 때는 `supply.sector.index_name_for()` 로 이름을 바꾸고,
"업종 상위 N" 을 고를 때는 `supply.sector.UMBRELLA_INDICES` 를 뺀다.

**파일은 HF 에 올리지 않는다** (약관 제12조 ②). 칸만 나간다.

In [10]:
conn.close()
print("끝")

끝
